In [1]:
import os
import zipfile

# Ensure kaggle CLI is installed
!pip install -q kaggle

# --- Instructions ---
# Please upload your 'kaggle.json' file to /content/ (or your current directory)
# or copy it to '~/.kaggle/kaggle.json' before running this cell.

# Set up directory for Kaggle credentials
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

# If kaggle.json is uploaded in current workspace, move it to the correct folder
if os.path.exists('kaggle.json'):
    os.rename('kaggle.json', os.path.join(kaggle_dir, 'kaggle.json'))
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

# Configuration
KAGGLE_DATASET = 'shrutimechlearn/churn-modelling'
TARGET_NAME = 'Churn_Modelling.csv'

# Download dataset zip using official Kaggle API
!kaggle datasets download -d {KAGGLE_DATASET}

# Extract zip and rename/move the csv file
zip_filename = KAGGLE_DATASET.split('/')[-1] + '.zip'
if os.path.exists(zip_filename):
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('temp_extracted')

    # Find the csv and place it at root
    for root, dirs, files in os.walk('temp_extracted'):
        for file in files:
            if file.endswith('.csv'):
                os.rename(os.path.join(root, file), TARGET_NAME)
                print(f'Successfully saved dataset as {TARGET_NAME}')
                break

    # Clean up temporary files
    os.remove(zip_filename)
    import shutil
    shutil.rmtree('temp_extracted')
else:
    print("Ensure you have uploaded kaggle.json so the API can download the dataset.")

Dataset URL: https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling
License(s): unknown
100% 262k/262k [00:00<00:00, 27.9MB/s]

Successfully saved dataset as Churn_Modelling.csv


In [10]:
# Bank Customer Churn Prediction using a Neural Network
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Read the dataset (Upload 'Churn_Modelling.csv' to Colab first)
df = pd.read_csv('Churn_Modelling.csv')

display(df.head(n=5))

# Pre-processing: Drop irrelevant identifier columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encode categorical text columns (Geography, Gender) into numbers
le = LabelEncoder()
df['Geography'] = le.fit_transform(df['Geography'])
df['Gender'] = le.fit_transform(df['Gender'])

# 2. Distinguish features and target
X = df.drop('Exited', axis=1)  # Features
y = df['Exited']               # Target (1 = Churned, 0 = Stayed)

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Normalize the train and test data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit and transform on train
X_test_scaled = scaler.transform(X_test)        # Only transform on test (prevent data leakage)

# 4. Initialize and build the Neural Network model
# Using MLPClassifier from scikit-learn.
# Improvement (Step 4): Adding hidden layers (e.g., 2 layers of 16 neurons),
# increasing max_iter to ensure convergence, and using 'relu' activation.
nn_model = MLPClassifier(hidden_layer_sizes=(16, 16, 8), max_iter=500)
nn_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = nn_model.predict(X_test_scaled)

# 5. Print Accuracy Score and Confusion Matrix
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("=== Neural Network Performance === ")
print(f"Accuracy Score: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nConfusion Matrix:")
print(cm)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


=== Neural Network Performance ===
Accuracy Score: 0.8605 (86.05%)

Confusion Matrix:
[[1524   83]
 [ 196  197]]
